# ASG Airlines - End-to-End Data Engineering Pipeline

This notebook walks through the pipeline stage by stage, showing the state of the
data before and after each one.

**It calls the modules in `src/` - it does not re-implement them.** There is exactly
one copy of every business rule, and it lives in the module that owns it. If a number
here disagrees with a number in the dashboard, the module is the source of truth.

Full reasoning for every choice below is in `DECISIONS.md`, referenced as D-0NN.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import clean, config, ingest, kpis, model, privacy, transform, validate

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

---
## 1. Ingest

Every column is read as **text**, deliberately. If pandas types the columns at read
time, a corrupt value is coerced to `NaN` before any code can record that it was
corrupt and why - `payments.amount` contains 30 cells holding the literal string
`"INVALID"`, and as a float they are indistinguishable from the 48 that are genuinely
blank. (D-002)

In [2]:
frames = ingest.load_workbook()

for name, df in frames.items():
    print(f"{name:<11} {len(df):>5} rows x {df.shape[1]} columns")

flights      1020 rows x 7 columns
bookings     1000 rows x 9 columns
passengers   1039 rows x 9 columns
payments     1000 rows x 4 columns


In [3]:
frames["flights"].head()

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701000,2026-04-21 02:32:41.701000,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703000,2026-04-21 01:23:41.703000,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702000,2026-04-21 01:11:41.702000,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704000,2026-04-21 01:43:41.704000,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703000,2026-04-21 04:04:41.703000,04:59:00


### Profiling the defects

Before changing anything, measure what is wrong. Every number below drives a
decision later in the notebook.

In [4]:
flights_raw = frames["flights"]

print("DEFECT INVENTORY")
print("-" * 62)
print(f"airline null or 'UNKNOWN'   : {flights_raw['airline'].isna().sum() + (flights_raw['airline'] == 'UNKNOWN').sum():>5}")
print(f"duplicate flight_id rows    : {flights_raw['flight_id'].duplicated().sum():>5}")
print(f"exact duplicate rows        : {flights_raw.duplicated().sum():>5}")
print(f"booking status null/INVALID : {frames['bookings']['status'].isna().sum() + (frames['bookings']['status'] == 'INVALID').sum():>5}")
print(f"payment amount unusable     : {pd.to_numeric(frames['payments']['amount'], errors='coerce').isna().sum():>5}")
print(f"passenger_id duplicated     : {frames['passengers']['passenger_id'].duplicated().sum():>5}")
print(f"payments {len(frames['payments'])} belong to only {frames['payments']['booking_id'].nunique()} bookings")

DEFECT INVENTORY
--------------------------------------------------------------
airline null or 'UNKNOWN'   :    72
duplicate flight_id rows    :    16
exact duplicate rows        :    15
booking status null/INVALID :    75
payment amount unusable     :    78
passenger_id duplicated     :    39
payments 1000 belong to only 637 bookings


**The timestamp format trap.** The source mixes two ISO layouts. Counting the
shapes shows it plainly - and shows why a naive parse would have quietly destroyed
31 valid flights. (D-005)

In [5]:
import re

def shape(value):
    return re.sub(r"\d", "9", str(value))

for col in ["departure_time", "arrival_time"]:
    print(col)
    print(flights_raw[col].map(shape).value_counts().to_string(), "\n")

departure_time
departure_time
9999-99-99 99:99:99.999999    1016
9999-99-99 99:99:99              4 

arrival_time
arrival_time
9999-99-99 99:99:99.999999    989
9999-99-99 99:99:99            31 



---
## 2. Validate the schema

A contract check. A missing column stops the run immediately rather than producing a
`KeyError` fifty lines deeper in the pipeline.

In [6]:
validate.reset()
validate.check_schema(frames)
print("Schema OK")

Schema OK


---
## 3. Clean

Null tokens, duplicates, standardisation, and the two repairs that recover data
rather than discarding it.

### The airline repair (D-003)

72 of 1020 flights have no usable airline. Rather than dropping 7% of the fleet or
inventing an "Unknown" carrier, the airline is recovered from the `flight_id`
prefix - but only because that mapping is provably 1:1.

In [7]:
prefix_check = pd.crosstab(flights_raw["flight_id"].str[:2],
                           flights_raw["airline"].fillna("<NULL>"))
prefix_check

airline,<NULL>,Air India,IndiGo,SpiceJet,UNKNOWN,Vistara
flight_id,,,,,,
6F,12,0,249,0,12,0
AI,15,236,0,0,9,0
SJ,5,0,0,240,6,0
UK,9,0,0,0,4,223


Every prefix maps to exactly one airline. `<NULL>` and `UNKNOWN` are the only
other entries in each row, so filling them from the prefix is lossless recovery, not
a guess. `repair_airline_from_flight_id` re-proves this on every run and **raises**
on an unseen prefix instead of defaulting.

In [8]:
flights = clean.clean_flights(frames["flights"])
passengers = clean.clean_passengers(frames["passengers"])
bookings = clean.clean_bookings(frames["bookings"])
payments = clean.clean_payments(frames["payments"])

print(f"\nflights    {len(frames['flights']):>5} -> {len(flights):>5}")
print(f"passengers {len(frames['passengers']):>5} -> {len(passengers):>5}")
print(f"bookings   {len(frames['bookings']):>5} -> {len(bookings):>5}")
print(f"payments   {len(frames['payments']):>5} -> {len(payments):>5}")

Quarantined 1 row(s) from flights: duplicate flight_id with conflicting attributes; first occurrence kept



flights     1020 ->  1004
passengers  1039 ->  1000
bookings    1000 ->  1000
payments    1000 ->  1000


In [9]:
flights[flights["airline_was_repaired"]][
    ["flight_id", "airline", "airline_was_repaired"]].head()

,flight_id,airline,airline_was_repaired
9,6F251,IndiGo,True
14,AI069,Air India,True
28,UK003,Vistara,True
57,UK209,Vistara,True
58,SJ043,SpiceJet,True


---
## 4. Transform - duration and overnight flights

The part the case study is really testing.

1. Parse both timestamps with `format="mixed"` so neither layout is lost.
2. If arrival is earlier than departure, the arrival **date** lost a day - add one.
3. `duration_minutes = arrival - departure`, after the repair.
4. Flag, never delete, anything still implausible.

In [10]:
flights = transform.transform_flights(flights)

print(f"\nflights crossing midnight : {flights['is_overnight'].sum()}")
print(f"arrival dates repaired    : {flights['arrival_date_was_repaired'].sum()}")
print(f"anomalies flagged         : {flights['is_anomaly'].sum()}")


flights crossing midnight : 122
arrival dates repaired    : 1
anomalies flagged         : 2


### Overnight flights get a correct positive duration

These depart late at night and land the next morning. Because the source timestamps
carry full dates, plain subtraction already handles them - they never need the +1 day
repair. (D-006)

In [11]:
flights[flights["is_overnight"]][
    ["flight_id", "airline", "route", "departure_time", "arrival_time",
     "duration_minutes", "is_overnight"]].head()

,flight_id,airline,route,departure_time,arrival_time,duration_minutes,is_overnight
0,SJ010,SpiceJet,CCU - MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,174,True
1,AI155,Air India,BOM - CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,108,True
2,UK094,Vistara,BOM - CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,105,True
3,AI245,Air India,BOM - CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,156,True
4,AI192,Air India,MAA - BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,299,True


### The one record that did need repairing (D-007)

`SJ192` departs on 19 April but records its arrival on the **18th** - 19 hours
before it left. Adding one day gives 300 minutes.

The repair is independently confirmed: the source workbook's own Excel formula
evaluates to `05:00:00` for this row - exactly 300 minutes. Two different tools
agreeing is far stronger evidence than the timestamp alone, which is why this record
was repaired rather than thrown away.

In [12]:
flights[flights["arrival_date_was_repaired"]][
    ["flight_id", "route", "departure_time", "arrival_time",
     "duration_minutes", "anomaly_reason"]]

,flight_id,route,departure_time,arrival_time,duration_minutes,anomaly_reason
350,SJ192,HYD - BOM,2026-04-19 18:45:42,2026-04-19 23:45:42,300,Arrival date repaired (cross-day)


### Referential integrity

Run *after* transform, because transform can quarantine a flight - which would
orphan any booking that referenced it.

In [13]:
bookings, payments = validate.check_referential_integrity(
    bookings, flights, passengers, payments)

print(f"bookings {len(bookings)}, payments {len(payments)} - no orphans")

bookings 1000, payments 1000 - no orphans


---
## 5. Protect PII

Four techniques, chosen per column by what the analysis actually needs. Hashing
everything would destroy email-domain and age analysis; masking everything would
break passenger counts. (D-015)

| Technique | Columns | Why |
|---|---|---|
| Salted SHA-256 | `aadhaar_id`, `passport_number`, `passenger_id` | Must still JOIN and COUNT DISTINCT |
| Partial mask | `email`, `phone` | An operator needs to verify a contact |
| Generalise | `date_of_birth` to `age_band` | An exact DOB re-identifies; a band does not |
| Drop | names, emergency contacts | They answer no KPI here |

In [14]:
print("BEFORE")
display(frames["passengers"][
    ["passenger_id", "first_name", "email", "phone", "aadhaar_id"]].head(3))

passengers = privacy.mask_passengers(passengers)
bookings = privacy.mask_bookings(bookings)

print("\nAFTER")
display(passengers.head(3))

BEFORE


,passenger_id,first_name,email,phone,aadhaar_id
0,P1000,Vivaan,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001
1,P1001,Krishna,krishna.reddy@hotmail.com,+91-6702632297,386379402654
2,P1002,Myra,myra.naidu@outlook.com,+91-6199585092,615594078161



AFTER


,passenger_id,passenger_key,aadhaar_hash,email_masked,email_domain,phone_masked,gender,age,age_band
0,P1000,4f13b2daa5295ad4,63525eecb990cb90,v***************e@gmail.com,gmail.com,+91-68****3790,F,52,45-54
1,P1001,fed8d21f8c78a611,d1cd5be7477b08c4,k***********y@hotmail.com,hotmail.com,+91-67****2297,M,15,13-17
2,P1002,36b6ee7ac73ef4b8,dbd9779f1abb4bdc,m********u@outlook.com,outlook.com,+91-61****5092,M,72,65+


The guard rail is not advisory - it raises and stops the run if any raw PII column
reaches a curated table.

In [15]:
privacy.assert_no_raw_pii(flights, bookings, passengers, payments)

---
## 6. Model - the star schema

Four dimensions, two facts.

**The trap this avoids:** 1,000 payments belong to only 637 bookings. Joining
payments straight onto bookings is one-to-many, so it duplicates booking rows and
silently inflates every count and revenue figure downstream - with no error message.
Payments are aggregated to booking grain *first*, which makes the join 1:1, and
`build_fact_booking` asserts the row count is unchanged. (D-016)

In [16]:
payment_summary = model.summarise_payments_by_booking(payments)

print(f"payments        : {len(payments)}")
print(f"booking totals  : {len(payment_summary)}")
print(f"bookings with >1 payment: {(payment_summary['payment_count'] > 1).sum()}")
payment_summary.sort_values("payment_count", ascending=False).head()

payments        : 1000
booking totals  : 637
bookings with >1 payment: 267


,booking_id,booking_total_amount,payment_count,payments_missing_amount
411,B1663,66885.47,6,0
566,B1895,54031.62,5,0
435,B1705,20858.79,4,0
98,B1165,43097.99,4,0
6,B1012,23702.39,4,1


In [17]:
tables = model.build_star_schema(flights, bookings, passengers, payments)
model.write_gold(tables)

---
## 7. KPIs

Nine pre-aggregated tables, each with a stated grain.

In [18]:
kpi_tables = kpis.build_all(tables)
kpis.write_all(kpi_tables)

In [19]:
kpi_tables["kpi_headline"].T.rename(columns={0: "value"})

,value
total_flights,1004.00
total_bookings,1000.00
total_passengers,636.00
airlines_operating,4.00
routes_operated,30.00
avg_flight_duration_minutes,164.80
median_flight_duration_minutes,166.50
overnight_flights,122.00
overnight_flight_pct,12.20
anomaly_flights,2.00


In [20]:
kpi_tables["kpi_duration_by_airline"]

,airline,flights_operated,avg_duration_minutes,median_duration_minutes,min_duration_minutes,max_duration_minutes,overnight_flights,anomalies,share_of_flights_pct
1,IndiGo,272,165.4,169.0,30,300,32,1,27.1
0,Air India,255,165.4,164.0,35,299,33,0,25.4
2,SpiceJet,247,163.8,166.0,32,300,29,1,24.6
3,Vistara,230,164.3,166.0,30,300,28,0,22.9


Average duration is nearly identical across all four carriers - 163.8 to 165.4
minutes. That evenness is itself the finding: no airline runs systematically longer
sectors on this network.

In [21]:
kpi_tables["kpi_route_traffic"].head(10)

,route,source,destination,flights_operated,avg_duration_minutes,overnight_flights,anomalies,bookings,revenue
6,BOM - CCU,BOM,CCU,90,169.5,16,0,87,700135.30
12,CCU - DEL,CCU,DEL,72,153.6,11,0,73,582851.37
25,MAA - BLR,MAA,BLR,65,172.8,8,0,64,492102.05
0,BLR - BOM,BLR,BOM,60,147.7,7,0,62,506069.04
24,HYD - MAA,HYD,MAA,57,152.8,5,0,57,438428.50
18,DEL - HYD,DEL,HYD,54,174.8,6,0,55,521798.78
23,HYD - DEL,HYD,DEL,42,185.4,9,0,41,305771.13
7,BOM - DEL,BOM,DEL,39,153.8,3,0,36,299355.97
11,CCU - BOM,CCU,BOM,33,164.0,1,0,34,150839.53
15,DEL - BLR,DEL,BLR,29,170.0,3,1,30,201095.92


In [22]:
kpi_tables["kpi_airport_traffic"]

,airport,departures,arrivals,total_movements
1,BOM,205,169,374
3,DEL,160,198,358
2,CCU,169,187,356
4,HYD,178,141,319
5,MAA,157,147,304
0,BLR,135,162,297


### Anomalies - and the honest gap on delays

The brief asks for delay analysis. **This dataset cannot support it.** A delay is
actual departure minus *scheduled* departure, and there is one departure timestamp
per flight with no schedule to compare against. Any "delay" computed here would be a
fabricated number presented as an operational fact.

What *is* measurable is duration anomaly detection, using a Tukey fence (1.5 x IQR)
applied per route. A fixed 90-minute tolerance was tried first and flagged 30% of the
fleet - because durations on every route naturally span 30-293 minutes. An anomaly
report that flags a third of your records is one nobody reads. (D-009, D-011)

In [23]:
kpi_tables["kpi_anomalies"]

,flight_id,airline,route,departure_time,arrival_time,duration_minutes,route_median_duration_minutes,duration_vs_route_median_minutes,is_overnight,anomaly_reason
0,SJ192,SpiceJet,HYD - BOM,2026-04-19 18:45:42.000,2026-04-19 23:45:42.000,300,150.0,150,False,Arrival date repaired (cross-day)
1,6F250,IndiGo,DEL - BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701,244,191.0,53,False,Conflicting duplicate flight_id


---
## 8. Reconciliation

Nothing is dropped silently. Every rejected row is in
`data/quarantine/quarantine.csv` with the rule that rejected it, and the totals must
balance:

    rows_in = rows_out + quarantined + deduplicated

In [24]:
rejected = validate.write_quarantine()

rows_in = 1020 + 1000 + 1039 + 1000
rows_out = (len(tables["dim_flight"]) + len(tables["fact_booking"])
            + len(tables["dim_passenger"]) + len(tables["fact_payment"]))

print(f"source rows   : {rows_in}")
print(f"curated rows  : {rows_out}")
print(f"quarantined   : {rejected}")
print(f"deduplicated  : {rows_in - rows_out - rejected}")

source rows   : 4059
curated rows  : 4004
quarantined   : 1
deduplicated  : 54


In [25]:
pd.read_csv(config.QUARANTINE_DIR / "quarantine.csv")[
    ["flight_id", "_source_sheet", "_rejection_reason"]]

,flight_id,_source_sheet,_rejection_reason
0,6F250,flights,duplicate flight_id with conflicting attribute...


---
## 9. The whole pipeline in one call

Everything above is what `pipeline.run()` does. In production this is the only entry
point:

```bash
python -m src.pipeline
```

Output lands in `data/gold/`, which is what the Power BI dashboard reads. See
`docs/powerbi_build_guide.md` for the dashboard build and `README.md` for the full
execution trace.

In [26]:
from src import pipeline

tables, kpi_tables = pipeline.run()

14:59:10  INFO    pipeline       ==============================================================================


14:59:10  INFO    pipeline       ASG Airlines pipeline starting


14:59:10  INFO    pipeline       ==============================================================================


14:59:10  INFO    pipeline       [1/7] Ingest


14:59:10  INFO    src.ingest     Reading workbook: C:\Users\DELL\OneDrive\Documents\projects\asg-airlines-pipeline\data\raw_UseCase-Airlines.xlsx


...

14:59:11  INFO    pipeline         curated fact/dim   : dim_flight 1004, fact_booking 1000, dim_passenger 1000, fact_payment 1000


14:59:11  INFO    pipeline         quarantined        : 1  (see data/quarantine/quarantine.csv)


14:59:11  INFO    pipeline         deduplicated       : 54


14:59:11  INFO    pipeline       ------------------------------------------------------------------------------


14:59:11  INFO    pipeline       Pipeline finished successfully
